# Generating Financial Sentiment Explanations

This notebook demonstrates how to generate reasoned financial sentiment explanations for the **FinancialPhraseBank** dataset. 

We use the **Qwen/Qwen2.5-7B-Instruct** model, which provides a strong balance of instruction following and financial reasoning capabilities. To ensure it fits within a 24GB VRAM GPU (like an RTX 3090/4090), we utilize 4-bit quantization (NF4).

In [ ]:
%%capture
# Install necessary libraries for model training and evaluation
import os
if "COLAB_" not in "".join(os.environ.keys()):
    pass
else:
    !pip install -U transformers trl accelerate bitsandbytes

In [ ]:
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import os

## Configuration

Here we define the model, dataset, and output settings. We use `BitsAndBytesConfig` for 4-bit quantization to optimize memory usage.

In [ ]:
MODEL_ID        = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID      = "lmassaron/FinancialPhraseBank"
OUTPUT_PATH     = "FinancialPhraseBank_explained"
PUSH_TO_REPO    = False
HF_REPO_ID      = None  # Set to "your-username/dataset-name" to push to Hub
BATCH_SIZE      = 16
MAX_NEW_TOKENS  = 512

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

LABEL_MAP = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

## Prompt Engineering

We define a `SYSTEM_PROMPT` that sets the persona of a senior financial analyst and a helper function to build the user prompt for each headline.

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior financial analyst with deep expertise in equity markets, "
    "corporate finance, and macroeconomics. "
    "Your task is to explain, in 2-4 sentences, why a financial news headline "
    "carries a specific market sentiment. "
    "Focus strictly on the financial implications: how the news affects revenue, "
    "profitability, cash flow, investor confidence, or market positioning. "
    "Be concise and precise. Do not repeat the sentence verbatim."
)

def build_prompt(sentence: str, sentiment: str) -> str:
    return (
        f"Financial news headline:\n\"{sentence}\"\n\n"
        f"This headline has been classified as **{sentiment}** sentiment "
        f"from a financial markets perspective.\n\n"
        f"Explain why, focusing on the financial implications for investors, "
        f"the company, or the broader market."
    )

## Model and Tokenizer Loading

We load the model with the previously defined quantization config and set up the tokenizer.

In [ ]:
def load_model(model_id: str):
    print(f"Loading tokenizer and model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    
    device_info = getattr(model, "hf_device_map", None) or str(model.device)
    print(f"Model loaded. Device: {device_info}")
    
    return tokenizer, model

tokenizer, model = load_model(MODEL_ID)

## Inference Logic

This function handles the batch generation of explanations. It uses the chat template provided by the model's tokenizer to format the prompts correctly.

In [ ]:
def generate_explanations_batch(
    tokenizer,
    model,
    sentences: list[str],
    labels: list[int],
) -> list[str]:
    """Run inference on a batch and return explanation strings."""

    # Build chat-formatted prompts
    chats = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_prompt(s, LABEL_MAP[l])},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for s, l in zip(sentences, labels)
    ]

    inputs = tokenizer(
        chats,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy — deterministic and fast
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the prompt)
    input_len = inputs["input_ids"].shape[1]
    explanations = tokenizer.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True,
    )
    return [e.strip() for e in explanations]

## Dataset Processing

We process the dataset in batches and collect the generated explanations into a new dataset format.

In [ ]:
def process_dataset(tokenizer, model, dataset) -> list[dict]:
    """Iterate over the dataset in batches, collecting results."""
    results = []
    sentences = dataset["sentence"]
    labels    = dataset["label"]
    n         = len(sentences)

    for start in tqdm(range(0, n, BATCH_SIZE), desc="Generating explanations"):
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels    = labels[start    : start + BATCH_SIZE]

        explanations = generate_explanations_batch(
            tokenizer, model, batch_sentences, batch_labels
        )

        for sentence, label, explanation in zip(
            batch_sentences, batch_labels, explanations
        ):
            results.append({
                "sentence":    sentence,
                "label":       label,
                "sentiment":   LABEL_MAP[label],
                "explanation": explanation,
            })

    return results

## Main Execution

Finally, we load the dataset, iterate through its splits, generate the explanations, and save the resulting `DatasetDict`.

In [ ]:
# 1. Load dataset
print(f"Loading dataset: {DATASET_ID}")
raw = load_dataset(DATASET_ID)
print(raw)

# 2. Process every split
output_splits = {}
for split_name, data in raw.items():
    print(f"\n── Processing split: '{split_name}' ({len(data)} examples) ──")
    results = process_dataset(tokenizer, model, data)
    output_splits[split_name] = Dataset.from_list(results)

output_ds = DatasetDict(output_splits)

# 3. Save locally
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nDataset saved to: {OUTPUT_PATH}")

## Verification

Let's look at one example from the generated dataset.

In [ ]:
first_split = list(output_splits.keys())[0]
ex = output_ds[first_split][0]
print(f"\nExample from '{first_split}':")
print(f"  Sentence  : {ex['sentence']}")
print(f"  Sentiment : {ex['sentiment']} (label={ex['label']})")
print(f"  Explanation:\n    {ex['explanation']}")

## Optional: Push to Hub

Uncomment and set your repository ID to push the generated dataset to the Hugging Face Hub.

In [ ]:
if HF_REPO_ID and PUSH_TO_REPO:
    print(f"Pushing to Hub: {HF_REPO_ID}")
    output_ds.push_to_hub(HF_REPO_ID)
    print("Upload complete!")